# 09 存活分析 — 參考解答

松柏護理之家退伍軍人症群聚事件存活分析練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1].copy()
cases["event"] = (cases["outcome"] == "dead").astype(int)

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

## 題目 1：CHF（心衰竭）的存活分析

In [ ]:
# KM 曲線：CHF vs No CHF
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("CHF", cases["comorbidity_chf"] == 1),
                     ("No CHF", cases["comorbidity_chf"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("存活曲線：CHF vs No CHF")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank 檢定
chf_yes = cases[cases["comorbidity_chf"] == 1]
chf_no = cases[cases["comorbidity_chf"] == 0]

result = logrank_test(
    chf_yes["time_to_event"], chf_no["time_to_event"],
    event_observed_A=chf_yes["event"],
    event_observed_B=chf_no["event"],
)

print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("\n\u2192 p < 0.05：CHF 顯著影響存活")
else:
    print("\n\u2192 p \u2265 0.05：CHF 對存活的影響未達統計顯著")
    print("\u2192 可能因為樣本數不足（死亡人數僅 19），統計檢定力不夠")

## 題目 2：年齡分組的存活比較

In [ ]:
# 年齡分組
cases["age_group"] = np.where(cases["age"] >= 75, "\u226575", "<75")

fig, ax = plt.subplots(figsize=(8, 5))

for label in ["\u226575", "<75"]:
    sub = cases[cases["age_group"] == label]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"Age {label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("存活曲線：Age \u226575 vs <75")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
old = cases[cases["age"] >= 75]
young = cases[cases["age"] < 75]

result_age = logrank_test(
    old["time_to_event"], young["time_to_event"],
    event_observed_A=old["event"],
    event_observed_B=young["event"],
)

print(f"Log-rank test statistic = {result_age.test_statistic:.3f}")
print(f"p-value = {result_age.p_value:.4f}")

if result_age.p_value < 0.05:
    print("\n\u2192 高齡（\u226575）組的存活顯著較差")
else:
    print("\n\u2192 年齡分組對存活的影響未達統計顯著")
    print("\u2192 護理之家住民普遍年齡較高，組間差異可能不夠大")

## 題目 3（挑戰題）：住院 vs 未住院 + Cox 迴歸

In [ ]:
# KM 曲線：住院 vs 未住院
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("住院", cases["hospitalized"] == 1),
                     ("未住院", cases["hospitalized"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("存活曲線：住院 vs 未住院")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
hosp_yes = cases[cases["hospitalized"] == 1]
hosp_no = cases[cases["hospitalized"] == 0]

result_hosp = logrank_test(
    hosp_yes["time_to_event"], hosp_no["time_to_event"],
    event_observed_A=hosp_yes["event"],
    event_observed_B=hosp_no["event"],
)

print(f"Log-rank test statistic = {result_hosp.test_statistic:.3f}")
print(f"p-value = {result_hosp.p_value:.4f}")

In [ ]:
# Cox 迴歸
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "hospitalized", "comorbidity_copd", "comorbidity_chf",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox 迴歸結果 ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

# HR 森林圖
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression \u2014 HR 森林圖")
plt.tight_layout()
plt.show()

print("\n=== 解讀 ===")
hosp_hr = cph.summary.loc["hospitalized", "exp(coef)"]
print(f"hospitalized HR = {hosp_hr:.3f}")
if hosp_hr > 1:
    print("\u2192 住院者 HR > 1，看似住院『增加』死亡風險")
    print("\u2192 但這不代表住院是危險因子！")
    print("\u2192 住院是因為病情嚴重，是 confounding by indication")
    print("\u2192 住院是嚴重度的『標記』，不是死亡的『原因』")
else:
    print("\u2192 住院者 HR < 1，調整其他因子後住院可能有保護效果")
    print("\u2192 但解讀仍需小心 confounding by indication")

### 解讀重點

- **CHF**：心衰竭可能增加死亡風險，但在小樣本中可能未達統計顯著
- **年齡**：護理之家住民年齡普遍偏高，組間差異可能不大
- **住院**：這是存活分析中經典的 **confounding by indication** 案例
  - 住院者的死亡率可能較高，但原因是「比較嚴重的人才會住院」
  - 住院本身是治療行為，應該降低死亡風險
  - 但在觀察性資料中，住院的 HR 可能 > 1，因為它是嚴重度的標記
- **限制**：本案僅 19 例死亡，Cox 模型中放太多變項容易過度配適（overfitting），建議每個事件至少 10 個，所以最多放 1-2 個變項較穩定